# 15 Comparacion final de registration

Este notebook junta los resultados de los tres pasos de registration que ya tenemos:

1. **Rigido por mascaras** (`12_registro_rigido_mascaras_todos.ipynb`)
2. **Afin por mascaras** (`13_registro_afin_mascaras_todos.ipynb`)
3. **No rigido por mapas de distancia + TV-L1** (`14_registro_no_rigido_mascaras_todos.ipynb`)

El objetivo no es calcular un registro nuevo, sino dejar una tabla final clara:

- que metodo queda como mejor resultado aceptado por sujeto,
- que metricas tiene cada etapa,
- que sujetos requieren revision visual,
- y donde estan los overlays/contornos finales.


In [ ]:
from pathlib import Path
import csv
import json
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


BASE_DIR = Path(r'Registration\DeepLearning')
REG_DIR = BASE_DIR / 'Tecnicas_registration'
RIGID_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_registro_rigido_mascaras'
AFFINE_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_registro_afin_mascaras'
NONRIGID_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_registro_no_rigido_mascaras'
OUTPUT_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_comparacion_final_registration'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Rigido:', RIGID_ROOT)
print('Afin:', AFFINE_ROOT)
print('No rigido:', NONRIGID_ROOT)
print('Salida:', OUTPUT_ROOT)


## Lectura de resumenes

Cada notebook anterior ya genero su CSV/JSON. Aqui los leemos y normalizamos nombres de columnas para poder comparar.


In [ ]:
def read_csv_dicts(path):
    with Path(path).open('r', newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def to_float(value, default=np.nan):
    if value is None or value == '':
        return default
    try:
        return float(value)
    except Exception:
        return default


def to_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {'true', '1', 'yes', 'si', 'sí'}


def index_by_subject(rows):
    return {row['subject_id']: row for row in rows}


rigid_rows = index_by_subject(read_csv_dicts(RIGID_ROOT / 'rigid_mask_summary.csv'))
affine_rows = index_by_subject(read_csv_dicts(AFFINE_ROOT / 'affine_mask_summary.csv'))
nonrigid_rows = index_by_subject(read_csv_dicts(NONRIGID_ROOT / 'nonrigid_mask_summary.csv'))

print('Sujetos rigido:', sorted(rigid_rows))
print('Sujetos afin:', sorted(affine_rows))
print('Sujetos no rigido:', sorted(nonrigid_rows))


## Tabla larga: una fila por sujeto y metodo

Esta tabla sirve para comparar directamente las metricas de cada metodo.


In [ ]:
def build_method_rows(subject_id):
    rigid = rigid_rows[subject_id]
    affine = affine_rows[subject_id]
    nonrigid = nonrigid_rows[subject_id]

    return [
        {
            'subject_id': subject_id,
            'method': 'rigid',
            'accepted': True,
            'decision_note': 'baseline_rigid',
            'dice': to_float(rigid['dice_rigid']),
            'iou': to_float(rigid['iou_rigid']),
            'contour_mean_px': to_float(rigid['contour_mean_distance_px']),
            'contour_p95_px': to_float(rigid['contour_p95_distance_px']),
            'flow_p95_px': np.nan,
            'flow_max_px': np.nan,
            'overlay_rgb_path': rigid['overlay_rgb_path'],
            'overlay_contours_path': rigid['overlay_contours_path'],
        },
        {
            'subject_id': subject_id,
            'method': 'affine_final',
            'accepted': to_bool(affine['affine_accepted']),
            'decision_note': affine['affine_final_note'],
            'dice': to_float(affine['dice_affine_final']),
            'iou': to_float(affine['iou_affine_final']),
            'contour_mean_px': to_float(affine['contour_mean_distance_affine_px']),
            'contour_p95_px': to_float(affine['contour_p95_distance_affine_px']),
            'flow_p95_px': np.nan,
            'flow_max_px': np.nan,
            'overlay_rgb_path': affine['overlay_rgb_path'],
            'overlay_contours_path': affine['overlay_contours_path'],
        },
        {
            'subject_id': subject_id,
            'method': 'nonrigid_final',
            'accepted': to_bool(nonrigid['nonrigid_accepted']),
            'decision_note': nonrigid['nonrigid_decision'],
            'dice': to_float(nonrigid['dice_nonrigid_final']),
            'iou': to_float(nonrigid['iou_nonrigid_final']),
            'contour_mean_px': to_float(nonrigid['contour_mean_distance_final_px']),
            'contour_p95_px': to_float(nonrigid['contour_p95_distance_final_px']),
            'flow_p95_px': to_float(nonrigid['flow_p95_displacement_px']),
            'flow_max_px': to_float(nonrigid['flow_max_displacement_px']),
            'overlay_rgb_path': nonrigid['overlay_rgb_path'],
            'overlay_contours_path': nonrigid['overlay_contours_path'],
        },
        {
            'subject_id': subject_id,
            'method': 'nonrigid_candidate',
            'accepted': False,
            'decision_note': 'candidate_before_filter',
            'dice': to_float(nonrigid['dice_nonrigid_candidate']),
            'iou': to_float(nonrigid['iou_nonrigid_candidate']),
            'contour_mean_px': to_float(nonrigid['contour_mean_distance_candidate_px']),
            'contour_p95_px': to_float(nonrigid['contour_p95_distance_candidate_px']),
            'flow_p95_px': to_float(nonrigid['flow_p95_displacement_px']),
            'flow_max_px': to_float(nonrigid['flow_max_displacement_px']),
            'overlay_rgb_path': nonrigid['overlay_rgb_candidate_path'],
            'overlay_contours_path': nonrigid['overlay_contours_candidate_path'],
        },
    ]


method_rows = []
for subject_id in SUBJECTS:
    method_rows.extend(build_method_rows(subject_id))

method_rows[:4]


## Decision final por sujeto

Regla usada:

1. Si el no rigido final fue aceptado, queda como resultado final.
2. Si no, pero el afin fue aceptado, queda afin.
3. Si no, queda rigido.

Ademas anadimos una bandera de revision visual. Aunque un metodo este aceptado, si el desplazamiento no rigido p95 es alto, conviene revisarlo manualmente.


In [ ]:
def visual_review_flag(final_method, nonrigid):
    if final_method != 'nonrigid_final':
        return 'review_rejected_nonrigid' if not to_bool(nonrigid['nonrigid_accepted']) else 'ok'

    flow_p95 = to_float(nonrigid['flow_p95_displacement_px'])
    flow_max = to_float(nonrigid['flow_max_displacement_px'])
    if flow_p95 >= 80 or flow_max >= 110:
        return 'review_high_nonrigid_displacement'
    if flow_p95 >= 60:
        return 'review_moderate_nonrigid_displacement'
    return 'ok'


def choose_final_row(subject_id):
    rigid = rigid_rows[subject_id]
    affine = affine_rows[subject_id]
    nonrigid = nonrigid_rows[subject_id]

    if to_bool(nonrigid['nonrigid_accepted']):
        final_method = 'nonrigid_final'
        source = nonrigid
        dice = to_float(nonrigid['dice_nonrigid_final'])
        iou = to_float(nonrigid['iou_nonrigid_final'])
        contour_mean = to_float(nonrigid['contour_mean_distance_final_px'])
        contour_p95 = to_float(nonrigid['contour_p95_distance_final_px'])
        overlay_rgb_path = nonrigid['overlay_rgb_path']
        overlay_contours_path = nonrigid['overlay_contours_path']
        final_note = nonrigid['nonrigid_decision']
    elif to_bool(affine['affine_accepted']):
        final_method = 'affine_final'
        source = affine
        dice = to_float(affine['dice_affine_final'])
        iou = to_float(affine['iou_affine_final'])
        contour_mean = to_float(affine['contour_mean_distance_affine_px'])
        contour_p95 = to_float(affine['contour_p95_distance_affine_px'])
        overlay_rgb_path = affine['overlay_rgb_path']
        overlay_contours_path = affine['overlay_contours_path']
        final_note = affine['affine_final_note']
    else:
        final_method = 'rigid'
        source = rigid
        dice = to_float(rigid['dice_rigid'])
        iou = to_float(rigid['iou_rigid'])
        contour_mean = to_float(rigid['contour_mean_distance_px'])
        contour_p95 = to_float(rigid['contour_p95_distance_px'])
        overlay_rgb_path = rigid['overlay_rgb_path']
        overlay_contours_path = rigid['overlay_contours_path']
        final_note = 'fallback_to_rigid'

    return {
        'subject_id': subject_id,
        'final_method': final_method,
        'final_note': final_note,
        'visual_review_flag': visual_review_flag(final_method, nonrigid),
        'dice_rigid': to_float(rigid['dice_rigid']),
        'dice_affine_final': to_float(affine['dice_affine_final']),
        'dice_nonrigid_final': to_float(nonrigid['dice_nonrigid_final']),
        'dice_nonrigid_candidate': to_float(nonrigid['dice_nonrigid_candidate']),
        'dice_final': dice,
        'iou_final': iou,
        'contour_mean_final_px': contour_mean,
        'contour_p95_final_px': contour_p95,
        'affine_accepted': to_bool(affine['affine_accepted']),
        'affine_note': affine['affine_final_note'],
        'nonrigid_accepted': to_bool(nonrigid['nonrigid_accepted']),
        'nonrigid_decision': nonrigid['nonrigid_decision'],
        'nonrigid_flow_p95_px': to_float(nonrigid['flow_p95_displacement_px']),
        'nonrigid_flow_max_px': to_float(nonrigid['flow_max_displacement_px']),
        'target_px_per_cm': to_float(rigid.get('target_px_per_cm')),
        'overlay_rgb_path': overlay_rgb_path,
        'overlay_contours_path': overlay_contours_path,
    }


final_rows = [choose_final_row(subject_id) for subject_id in SUBJECTS]
final_rows


## Guardar CSV/JSON finales

In [ ]:
def json_ready(value):
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, dict):
        return {k: json_ready(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_ready(v) for v in value]
    return value


def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            clean = {}
            for key in fieldnames:
                value = row.get(key, '')
                if isinstance(value, float) and np.isnan(value):
                    value = ''
                clean[key] = value
            writer.writerow(clean)


method_fields = [
    'subject_id', 'method', 'accepted', 'decision_note',
    'dice', 'iou', 'contour_mean_px', 'contour_p95_px',
    'flow_p95_px', 'flow_max_px',
    'overlay_rgb_path', 'overlay_contours_path',
]
final_fields = [
    'subject_id', 'final_method', 'final_note', 'visual_review_flag',
    'dice_rigid', 'dice_affine_final', 'dice_nonrigid_final', 'dice_nonrigid_candidate',
    'dice_final', 'iou_final', 'contour_mean_final_px', 'contour_p95_final_px',
    'affine_accepted', 'affine_note',
    'nonrigid_accepted', 'nonrigid_decision',
    'nonrigid_flow_p95_px', 'nonrigid_flow_max_px',
    'target_px_per_cm',
    'overlay_rgb_path', 'overlay_contours_path',
]

method_csv = OUTPUT_ROOT / 'registration_methods_comparison_long.csv'
final_csv = OUTPUT_ROOT / 'registration_final_decision_summary.csv'
method_json = OUTPUT_ROOT / 'registration_methods_comparison_long.json'
final_json = OUTPUT_ROOT / 'registration_final_decision_summary.json'

write_csv(method_csv, method_rows, method_fields)
write_csv(final_csv, final_rows, final_fields)
method_json.write_text(json.dumps(json_ready(method_rows), indent=2), encoding='utf-8')
final_json.write_text(json.dumps(json_ready(final_rows), indent=2), encoding='utf-8')

print('Tabla larga CSV:', method_csv)
print('Decision final CSV:', final_csv)
print('Tabla larga JSON:', method_json)
print('Decision final JSON:', final_json)


## Resumen en pantalla

In [ ]:
for row in final_rows:
    print(
        f"{row['subject_id']}: final={row['final_method']} | "
        f"Dice={row['dice_final']:.4f} | IoU={row['iou_final']:.4f} | "
        f"contour_mean={row['contour_mean_final_px']:.2f}px | "
        f"flag={row['visual_review_flag']}"
    )


## Graficas comparativas

In [ ]:
subjects = [row['subject_id'] for row in final_rows]
x = np.arange(len(subjects))
width = 0.22

dice_rigid = [row['dice_rigid'] for row in final_rows]
dice_affine = [row['dice_affine_final'] for row in final_rows]
dice_nonrigid = [row['dice_nonrigid_final'] for row in final_rows]
dice_final = [row['dice_final'] for row in final_rows]
flow_p95 = [row['nonrigid_flow_p95_px'] for row in final_rows]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)

axes[0].bar(x - width, dice_rigid, width, label='rigido')
axes[0].bar(x, dice_affine, width, label='afin final')
axes[0].bar(x + width, dice_nonrigid, width, label='no rigido final')
axes[0].plot(x, dice_final, color='black', marker='o', linewidth=1.5, label='decision final')
axes[0].set_xticks(x)
axes[0].set_xticklabels(subjects)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Dice')
axes[0].set_title('Comparacion Dice por metodo')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25)

axes[1].bar(x, flow_p95, color='#4C78A8')
axes[1].axhline(60, color='#E45756', linestyle='--', linewidth=1, label='revision moderada')
axes[1].axhline(80, color='#B00020', linestyle='--', linewidth=1, label='revision alta')
axes[1].set_xticks(x)
axes[1].set_xticklabels(subjects)
axes[1].set_ylabel('p95 desplazamiento no rigido (px)')
axes[1].set_title('Magnitud de deformacion no rigida')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.25)

chart_path = OUTPUT_ROOT / 'registration_metrics_summary.png'
fig.savefig(chart_path, dpi=160, bbox_inches='tight')
plt.show()

print('Grafica resumen:', chart_path)


## Montajes visuales

Generamos tres montajes:

- comparacion de contornos por metodo,
- comparacion de overlays por metodo,
- resultado final seleccionado por sujeto.


In [ ]:
def load_image(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)


def resize_for_montage(img, target_width=300):
    h, w = img.shape[:2]
    scale = target_width / max(w, 1)
    target_h = max(1, int(round(h * scale)))
    return cv2.resize(img, (target_width, target_h), interpolation=cv2.INTER_AREA)


def add_label(img, label):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 32), (0, 0, 0), -1)
    cv2.putText(out, label, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def make_grid_from_paths(items, cols=3, target_width=300):
    images = []
    for path, label in items:
        img = load_image(path)
        images.append(add_label(resize_for_montage(img, target_width), label))

    rows = int(math.ceil(len(images) / cols))
    max_h = max(img.shape[0] for img in images)
    canvas = np.zeros((rows * max_h, cols * target_width, 3), dtype=np.uint8)
    for idx, img in enumerate(images):
        r = idx // cols
        c = idx % cols
        y = r * max_h
        x = c * target_width
        canvas[y:y + img.shape[0], x:x + img.shape[1]] = img
    return canvas


method_order = ['rigid', 'affine_final', 'nonrigid_final']
method_label = {
    'rigid': 'rigid',
    'affine_final': 'afin',
    'nonrigid_final': 'no rigido',
}

contour_items = []
overlay_items = []
for subject_id in SUBJECTS:
    by_method = {row['method']: row for row in method_rows if row['subject_id'] == subject_id}
    for method in method_order:
        row = by_method[method]
        label = f"{subject_id} | {method_label[method]} | D {row['dice']:.3f}"
        contour_items.append((row['overlay_contours_path'], label))
        overlay_items.append((row['overlay_rgb_path'], label))

final_contour_items = [
    (
        row['overlay_contours_path'],
        f"{row['subject_id']} | {row['final_method'].replace('_final', '')} | D {row['dice_final']:.3f}"
    )
    for row in final_rows
]
final_overlay_items = [
    (
        row['overlay_rgb_path'],
        f"{row['subject_id']} | {row['final_method'].replace('_final', '')} | {row['visual_review_flag']}"
    )
    for row in final_rows
]

contour_grid = make_grid_from_paths(contour_items, cols=3, target_width=320)
overlay_grid = make_grid_from_paths(overlay_items, cols=3, target_width=320)
final_contour_grid = make_grid_from_paths(final_contour_items, cols=3, target_width=360)
final_overlay_grid = make_grid_from_paths(final_overlay_items, cols=3, target_width=360)

contour_grid_path = OUTPUT_ROOT / 'registration_methods_contour_comparison_grid.png'
overlay_grid_path = OUTPUT_ROOT / 'registration_methods_overlay_comparison_grid.png'
final_contour_grid_path = OUTPUT_ROOT / 'registration_final_selected_contour_grid.png'
final_overlay_grid_path = OUTPUT_ROOT / 'registration_final_selected_overlay_grid.png'

Image.fromarray(contour_grid).save(contour_grid_path)
Image.fromarray(overlay_grid).save(overlay_grid_path)
Image.fromarray(final_contour_grid).save(final_contour_grid_path)
Image.fromarray(final_overlay_grid).save(final_overlay_grid_path)

print('Comparacion contornos:', contour_grid_path)
print('Comparacion overlays:', overlay_grid_path)
print('Final contornos:', final_contour_grid_path)
print('Final overlays:', final_overlay_grid_path)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
axes[0].imshow(final_overlay_grid)
axes[0].axis('off')
axes[0].set_title('Resultado final seleccionado - overlays')
axes[1].imshow(final_contour_grid)
axes[1].axis('off')
axes[1].set_title('Resultado final seleccionado - contornos')
plt.show()


## Lectura recomendada

La decision final automatica es util para ordenar el pipeline, pero no sustituye la validacion visual:

- `ok`: resultado razonable segun metrica y desplazamiento.
- `review_moderate_nonrigid_displacement`: no rigido aceptado, pero conviene mirar overlay/contorno.
- `review_high_nonrigid_displacement`: aceptado con desplazamiento grande; revisar con cuidado.
- `review_rejected_nonrigid`: el no rigido se rechazo y el final vuelve a afin/rigido.

Esta tabla sera la base para decidir si probar B-spline/Demons o si empezar a evaluar estructuras internas.
